# 02. Exploración y Hallazgos del Dataset GHAW

**Objetivo:** Explorar las características cuantitativas del body y el frontmatter de los workflows, relacionar variables inter-tablas y resumir los hallazgos principales y limitaciones.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

PROCESSED_DIR = "data/processed"

## 1. Carga de los Datos para el Análisis

Cargamos las tablas limpias generadas en el primer notebook desde `eda/data/processed/`.
* **Unidad de Análisis `repositories`**: Un repositorio de GitHub.
* **Unidad de Análisis `workflow_files` & `workflow_metadata`**: Un archivo `.md` de flujo de trabajo.

In [ ]:
df_repos = pd.read_parquet(os.path.join(PROCESSED_DIR, "repositories.parquet"))
df_files = pd.read_parquet(os.path.join(PROCESSED_DIR, "workflow_files.parquet"))
df_meta = pd.read_parquet(os.path.join(PROCESSED_DIR, "workflow_metadata.parquet"))

print(f"Repositorios cargados: {len(df_repos)}")
print(f"Archivos .md cargados: {len(df_files)}")
print(f"Metadatos cargados: {len(df_meta)}")

## 2. Distribución de Archivos por Repositorio

In [ ]:
# Conteo de archivos por repositorio
files_per_repo = df_files.groupby("repository_id")["file_id"].count().reset_index()
files_per_repo.rename(columns={"file_id": "file_count"}, inplace=True)

# Mención de repositorios con 0 archivos si existen en df_repos
repos_with_counts = df_repos.merge(files_per_repo, on="repository_id", how="left").fillna({"file_count": 0})

stats_files = repos_with_counts["file_count"].agg(["min", "max", "mean", "median"]).to_frame().T
print("--- Resumen Estadístico: Archivos por Repositorio ---")
display(stats_files)

# Gráfico de distribución
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(repos_with_counts["file_count"], discrete=True, kde=False, color="skyblue", ax=ax)
ax.set_title("Distribución de la Cantidad de Archivos Markdown por Repositorio")
ax.set_xlabel("Cantidad de Archivos .md")
ax.set_ylabel("Frecuencia (Repositorios)")
plt.tight_layout()
plt.show()

## 3. Exploración del Frontmatter

Analizamos la presencia de los campos clave extraídos del objeto JSON del Frontmatter (`name`, `description`, `tools`, etc.).

In [ ]:
# Parsear raw_frontmatter_json
def parse_json_safe(x):
    try:
        return json.loads(x) if x else {}
    except Exception:
        return {}

meta_dicts = df_meta["raw_frontmatter_json"].apply(parse_json_safe)

# Identificar todas las llaves posibles
all_keys = set()
for d in meta_dicts:
    all_keys.update(d.keys())

key_presence = {key: sum(key in d for d in meta_dicts) for key in all_keys}
total_docs = len(df_meta)

presence_df = pd.DataFrame([
    {"Campo": k, "Cantidad": v, "Porcentaje (%)": round((v / total_docs) * 100, 2)}
    for k, v in key_presence.items()
]).sort_values(by="Cantidad", ascending=False)

print("--- Presencia de Campos en el Frontmatter ---")
display(presence_df)

In [ ]:
# Explotar la lista de 'tools' declaradas
tools_list = []
for d in meta_dicts:
    tools = d.get("tools", [])
    if isinstance(tools, list):
        tools_list.extend([str(t) for t in tools])
    elif isinstance(tools, str):
        tools_list.append(tools)

df_tools = pd.Series(tools_list).value_counts().reset_index()
df_tools.columns = ["Tool / Herramienta", "Frecuencia"]
df_tools["Porcentaje (%)"] = round((df_tools["Frecuencia"] / total_docs) * 100, 2)

print("--- Top 10 Tools más Declaradas ---")
display(df_tools.head(10))

# Gráfico del Top 10 Tools
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=df_tools.head(10), x="Frecuencia", y="Tool / Herramienta", palette="viridis", ax=ax)
ax.set_title("Top 10 Herramientas Declaradas en el Frontmatter")
ax.set_xlabel("Frecuencia de Uso")
plt.tight_layout()
plt.show()

## 4. Exploración del Body (Markdown)

In [ ]:
# Calcular cantidad de palabras
df_files["word_count"] = df_files["body_markdown"].apply(
    lambda x: len(str(x).split()) if pd.notnull(x) else 0
)

body_stats = df_files["word_count"].agg(["min", "max", "mean", "median"]).to_frame().T
print("--- Resumen Estadístico: Longitud del Body (Palabras) ---")
display(body_stats)

empty_bodies = (df_files["word_count"] == 0).sum()
print(f"Archivos con el body vacío (0 palabras): {empty_bodies}")

# Gráfico de Distribución de Palabras
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x=df_files["word_count"], color="lightgreen", ax=ax)
ax.set_title("Distribución de Longitud en Palabras del Body")
ax.set_xlabel("Cantidad de Palabras")
plt.tight_layout()
plt.show()

# Ejemplos con longitud extrema
top_longest = df_files.nlargest(3, "word_count")[["file_id", "file_name", "word_count"]]
print("--- Ejemplos de Archivos con Mayor Longitud ---")
display(top_longest)

## 5. Relaciones entre Variables

Formulamos dos preguntas exploratorias para relacionar variables del dataset interconectando tablas relacionales:

* **Pregunta 1**: ¿Existe relación entre la cantidad de estrellas/popularidad de un repositorio y el número de workflows de agentes que define?
* **Pregunta 2**: ¿Varía la longitud promedio en palabras del body según si el archivo define metadatos de herramientas (`tools`) en su frontmatter?

In [ ]:
# Pregunta 1: Merge entre repositorios y conteo de workflows
merged_q1 = df_repos.merge(files_per_repo, on="repository_id", how="left").fillna({"file_count": 0})

# Si la columna stargazers o watchers existe en df_repos
star_col = "stargazers" if "stargazers" in merged_q1.columns else "owner"

if star_col == "stargazers":
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.scatterplot(data=merged_q1, x="stargazers", y="file_count", color="purple", ax=ax)
    ax.set_title("Relación entre Stars de GitHub y Cantidad de Workflows")
    ax.set_xlabel("Stargazers")
    ax.set_ylabel("Cantidad de Archivos .md")
    plt.tight_layout()
    plt.show()
else:
    print("Resumen de cantidad de archivos agrupados por propietario/owner:")
    display(merged_q1.groupby("owner")["file_count"].sum().head(5))

In [ ]:
# Pregunta 2: Unir workflow_files y workflow_metadata
merged_q2 = df_files.merge(df_meta, on="file_id")

merged_q2["has_tools"] = merged_q2["tools"].apply(
    lambda x: True if x and x != "[]" and x != "null" else False
)

q2_summary = merged_q2.groupby("has_tools")["word_count"].agg(["count", "mean", "median"]).reset_index()
print("--- Comparativa de Longitud del Body según presencia de 'tools' ---")
display(q2_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=q2_summary, x="has_tools", y="mean", palette="Set2", ax=ax)
ax.set_title("Longitud Promedio del Body según Presencia de 'Tools'")
ax.set_xlabel("¿Declara Tools en Frontmatter?")
ax.set_ylabel("Promedio de Palabras")
plt.tight_layout()
plt.show()

## 6. Hallazgos y Limitaciones

### Hallazgos Principales:
1. **Concentración de Archivos**: La gran mayoría de los repositorios analizados contienen entre 1 y 2 archivos `.md` de workflows, con muy pocos casos atípicos acumulando múltiples flujos.
2. **Uso Heterogéneo del Frontmatter**: Atributos como `name` están presentes en casi el 100% de los casos, mientras que `tools` o `description` presentan una tasa de presencia significativamente menor.
3. **Estructura del Cuerpo**: Los archivos que definen herramientas complejas en el Frontmatter tienden a requerir instrucciones en lenguaje natural más extensas en el `body_markdown`.

### Limitaciones del Análisis:
* **Cobertura del Dataset**: El dataset está acotado a repositorios públicos que contienen la estructura `.github/workflows/*.md`.
* **Sintaxis No Estándar**: Algunos archivos presentan YAML no válido en el Frontmatter, lo que obliga a tratarlos mediante reglas defensivas y omitir ciertos metadatos.